[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/Rang/blob/main/tools/notebooks/05_check_palette.ipynb)

<div style="font-family:Arial,sans-serif">

# 05. Check the palette

Review source distance, color separation, and the suggested categorical pick order.

Created by **Mohsen Tahmasebi Nasab, PhD**<br>
[hydromohsen.com](https://hydromohsen.com)

Copyright and license holder: Mohsen Tahmasebi Nasab. Notebook code is
licensed under the repository's MIT License. Rang palette data follows the
CC0 dedication described in the licensing guide. Source images keep their own
rights and reuse terms.

</div>

<div style="font-family:Arial,sans-serif;background:#fff3cd;padding:14px"><strong>YOUR INPUT</strong><br>Set the Git branch, choose whether to use Google Drive, and give the palette a short filename.</div>

In [ ]:
REPO_REF = "main" #@param {type:"string"}
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
PALETTE_SLUG = "your-palette" #@param {type:"string"}

Google Drive keeps the recipe available when you move to the next notebook.
For a local Jupyter session, working files are placed under
`cache/notebook_workflow/`. When testing a GitHub branch, replace `main` with
the branch name above.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    REPO_ROOT = Path("/content/Rang")
    if not (REPO_ROOT / "tools" / "notebook_workflow.py").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", REPO_REF,
            "https://github.com/mohsennasab/Rang.git", str(REPO_ROOT)
        ], check=True)
else:
    probe = Path.cwd().resolve()
    REPO_ROOT = next(
        candidate for candidate in (probe, *probe.parents)
        if (candidate / "tools" / "notebook_workflow.py").exists()
    )

try:
    import matplotlib
    import numpy
    import PIL
    import sklearn
except ImportError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(REPO_ROOT / "tools" / "requirements.txt")
    ], check=True)

sys.path.insert(0, str(REPO_ROOT / "tools"))

from notebook_workflow import *

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Rang") / PALETTE_SLUG
else:
    WORK_DIR = REPO_ROOT / "cache" / "notebook_workflow" / PALETTE_SLUG

WORK_DIR.mkdir(parents=True, exist_ok=True)
RECIPE_PATH = WORK_DIR / f"{PALETTE_SLUG}-recipe.json"
use_arial()
print("Repository:", REPO_ROOT)
print("Working folder:", WORK_DIR)
print("Recipe:", RECIPE_PATH)

In [ ]:
if not RECIPE_PATH.exists():
    raise FileNotFoundError(
        f"Recipe not found at {RECIPE_PATH}. Run notebook 01 first and use "
        "the same Google Drive and palette slug settings."
    )
recipe = read_json(RECIPE_PATH)
print(f'Loaded {recipe["palette"]} with {len(recipe["regions"])} regions')

In [ ]:
report = check_recipe(
    RECIPE_PATH, WORK_DIR, WORK_DIR / "check-report.json"
)
print(report["palette"], *report["colors"])
print("Suggested pick order:", report["suggested_pick_order"])
print("Color vision flag:", report["colorblind"])
print()
for view, values in report["viewing"].items():
    print(f'{view:14s} min={values["minimum"]:.1f} mean={values["mean"]:.1f}')
print()
for color, values in report["source_presence"].items():
    marker = "check" if values["nearest"] > 3 else ""
    print(color, f'nearest={values["nearest"]:.1f}', marker)
print("Saved report:", WORK_DIR / "check-report.json")

<div style="font-family:Arial,sans-serif;background:#d9edf7;padding:14px"><strong>YOUR DECISION</strong><br>Read the report and look at real sample plots. Keep meaningful colors when they serve the artwork, and explain any large source distance in the pull request.</div>

A number can point to a problem, but it cannot decide whether a palette feels
right. Return to notebooks 03 and 04 when two colors collapse together, a
color drifts too far from the source, or the ramp loses the mood of the work.